# Task 12 — LibreYOLO License Audit & Smoke

**Dataset:** Smoke only — no GT dataset

**Protocol:** smoke

**Models:** All LibreYOLO weights (44 default-safe, 14 blocked)

In [ ]:
# v2.47.2: LibreYOLO model coverage.
LIBREYOLO_MODELS = [
    "libreyolo-dfine-n", "libreyolo-dfine-s",
    "libreyolo-yolox-n", "libreyolo-yolox-s",
]
print(f"LibreYOLO model scope: {len(LIBREYOLO_MODELS)} models")


In [ ]:
import json, sys, os
from pathlib import Path

# Add shared dir to path
NB_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/home/arash/PycharmProjects/VisionServeX/notebook")
sys.path.insert(0, str(NB_ROOT / "shared"))
os.chdir(str(NB_ROOT.parent))

from paths import COCO_400_ANN, COCO_400_IMAGES, SMOKE_IMG, SMOKE_ANN, NB_ROOT, REPO_ROOT
from display import clean, scan_text
from commands import run
from notebook_utils import write_status

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import visionservex
print(f"VisionServeX {visionservex.__version__}")

TASK = NB_ROOT / '12_libreyolo'


In [ ]:
r = run(["libreyolo","license-audit","--format","json",
         "--out",str(TASK/"reports/libreyolo_license_audit.json")], timeout=30)
if r.get("_returncode")==0:
    rows = r.get("rows",[])
    print(f"LibreYOLO version : {r.get('libreyolo_version','?')}")
    df_l = pd.DataFrame(rows)
    if len(df_l):
        df_l.to_csv(TASK/"reports/libreyolo_families.csv", index=False)
        print(df_l[["family","weight_license","license_risk","auto_pull"]].to_string(index=False))
else:
    print(f"LibreYOLO: {r.get('code','LIBREYOLO_REQUIRED')}")

# Smoke test a default-safe model
r2 = run(["libreyolo","smoke-test","libreyolo-yolox-n",
          str(NB_ROOT / ".." / "tests/assets/smoke/coco_person_car.jpg"),
          "--device","cpu","--format","json",
          "--out",str(TASK/"reports/libreyolo_yolox_n_smoke.json")], timeout=90)
state = "smoke_passed" if r2.get("status")=="ok" else "expected_blocker"
print(f"libreyolo-yolox-n smoke: {state}")
pd.DataFrame([{"model_id":"libreyolo-yolox-n","final_state":state,"code":r2.get("code","-")}]).to_csv(
    TASK/"reports/libreyolo_smoke.csv",index=False)


In [ ]:
write_status(TASK, task='12_libreyolo', dataset='Smoke only — no GT dataset', protocol='smoke')
print('Status written.')